In [1]:
import json, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DATA_DIR = Path("data")
ARTIFACTS_DIR = Path("artifacts")
DATA_DIR.mkdir(exist_ok=True)
ARTIFACTS_DIR.mkdir(exist_ok=True)

DATA_URL = ("https://raw.githubusercontent.com/propublica/compas-analysis/"
            "master/compas-scores-two-years.csv")
DATA_PATH = DATA_DIR / "compas-scores-two-years.csv"

In [2]:

if not DATA_PATH.exists():
    pd.read_csv(DATA_URL).to_csv(DATA_PATH, index=False)

raw = pd.read_csv(DATA_PATH, parse_dates=["compas_screening_date"])
print("Filas originales:", raw.shape)

# Filtros elegidos: arresto a ±30 días del cribado y sin cargos tipo "O".
# NO filtramos por is_recid (usaría información posterior a la decisión).
df = raw[
    raw["days_b_screening_arrest"].between(-30, 30)
    & (raw["c_charge_degree"] != "O")
].copy()
print("Filas tras filtros:", df.shape)

# Comprobaciones rápidas
print("\nRango de fechas:", df["compas_screening_date"].min(), "->", df["compas_screening_date"].max())
print("\nTasa de reincidencia (two_year_recid):", df["two_year_recid"].mean().round(3))
print("\nNulos en las columnas que usaremos:")
cols = ["sex", "age", "priors_count", "juv_fel_count", "juv_misd_count", "juv_other_count",
        "c_charge_degree", "c_charge_desc", "days_b_screening_arrest", "c_days_from_compas",
        "race", "decile_score", "two_year_recid", "compas_screening_date"]
print(df[cols].isna().sum())
print("\nFilas con is_recid == -1 (seguimiento dudoso):", (df["is_recid"] == -1).sum())
print("\nRaza:\n", df["race"].value_counts())

Filas originales: (7214, 53)
Filas tras filtros: (6172, 53)

Rango de fechas: 2013-01-01 00:00:00 -> 2014-12-31 00:00:00

Tasa de reincidencia (two_year_recid): 0.455

Nulos en las columnas que usaremos:
sex                        0
age                        0
priors_count               0
juv_fel_count              0
juv_misd_count             0
juv_other_count            0
c_charge_degree            0
c_charge_desc              5
days_b_screening_arrest    0
c_days_from_compas         0
race                       0
decile_score               0
two_year_recid             0
compas_screening_date      0
dtype: int64

Filas con is_recid == -1 (seguimiento dudoso): 0

Raza:
 race
African-American    3175
Caucasian           2103
Hispanic             509
Other                343
Asian                 31
Native American       11
Name: count, dtype: int64


In [5]:
# Recorte por calidad de datos: desde abril de 2014 el archivo casi no contiene
# no reincidentes (seguimiento de 2 años incompleto), así que esos meses se excluyen.
CUTOFF = pd.Timestamp("2014-04-01")
n_before = len(df)
df = df[df["compas_screening_date"] < CUTOFF].copy()
print(f"Filas tras recorte temporal: {len(df)} (se pierden {n_before - len(df)})")

df = df.sort_values("compas_screening_date").reset_index(drop=True)
dates = df["compas_screening_date"]

cut_train = dates.iloc[int(0.6 * len(df))]
cut_val = dates.iloc[int(0.8 * len(df))]

train = df[dates < cut_train].copy()
val = df[(dates >= cut_train) & (dates < cut_val)].copy()
test = df[dates >= cut_val].copy()

assert len(train) + len(val) + len(test) == len(df)
assert train["compas_screening_date"].max() < val["compas_screening_date"].min()
assert val["compas_screening_date"].max() < test["compas_screening_date"].min()

for name, part in [("train", train), ("val", val), ("test", test)]:
    print(f"{name:5s} n={len(part):5d}  "
          f"{part['compas_screening_date'].min().date()} -> {part['compas_screening_date'].max().date()}  "
          f"reincidencia={part['two_year_recid'].mean():.3f}  "
          f"no_reincidentes={(part['two_year_recid'] == 0).sum()}")

split_label = pd.Series(
    np.where(dates < cut_train, "train", np.where(dates < cut_val, "val", "test")),
    index=df.index, name="split")
print("\nRaza por conjunto:")
print(pd.crosstab(df["race"], split_label))

Filas tras recorte temporal: 5297 (se pierden 875)
train n= 3175  2013-01-01 -> 2013-09-21  reincidencia=0.370  no_reincidentes=2000
val   n= 1061  2013-09-22 -> 2013-12-22  reincidencia=0.359  no_reincidentes=680
test  n= 1061  2013-12-23 -> 2014-03-31  reincidencia=0.359  no_reincidentes=680

Raza por conjunto:
split             test  train  val
race                              
African-American   521   1646  510
Asian                4     16    7
Caucasian          382   1063  383
Hispanic            94    270   83
Native American      0      6    3
Other               60    174   75


In [4]:
monthly = (df.assign(month=df["compas_screening_date"].dt.to_period("M"))
             .groupby("month")["two_year_recid"]
             .agg(n="size", tasa="mean", reincidentes="sum"))
monthly["no_reincidentes"] = monthly["n"] - monthly["reincidentes"]
print(monthly.round(3))

           n   tasa  reincidentes  no_reincidentes
month                                             
2013-01  505  0.388           196              309
2013-02  507  0.391           198              309
2013-03  485  0.388           188              297
2013-04  503  0.372           187              316
2013-05  403  0.362           146              257
2013-06   58  0.241            14               44
2013-07   72  0.319            23               49
2013-08  395  0.354           140              255
2013-09  367  0.351           129              238
2013-10  362  0.315           114              248
2013-11  308  0.399           123              185
2013-12  380  0.355           135              245
2014-01  339  0.357           121              218
2014-02  316  0.348           110              206
2014-03  297  0.380           113              184
2014-04   89  0.966            86                3
2014-05   93  1.000            93                0
2014-06   93  1.000            

In [6]:
def rate_by_bin(frame, col, bins, labels):
    b = pd.cut(frame[col], bins=bins, labels=labels, right=False)
    out = frame.groupby(b, observed=True)["two_year_recid"].agg(n="size", tasa="mean")
    return out.round(3)

print("Reincidencia por tramo de edad (train):")
print(rate_by_bin(train, "age", [0, 23, 31, 46, 200], ["<23", "23-30", "31-45", "46+"]))

print("\nReincidencia por tramo de antecedentes (train):")
print(rate_by_bin(train, "priors_count", [0, 1, 3, 6, 1000], ["0", "1-2", "3-5", "6+"]))

print("\nJuvenil (cualquier juv_*_count > 0) (train):")
has_juv = (train[["juv_fel_count", "juv_misd_count", "juv_other_count"]].sum(axis=1) > 0)
print(train.groupby(has_juv)["two_year_recid"].agg(n="size", tasa="mean").round(3))

print("\nTipo de cargo (train):")
print(train.groupby("c_charge_degree")["two_year_recid"].agg(n="size", tasa="mean").round(3))

print("\nSexo (train):")
print(train.groupby("sex")["two_year_recid"].agg(n="size", tasa="mean").round(3))

Reincidencia por tramo de edad (train):
          n   tasa
age               
<23     311  0.482
23-30  1167  0.444
31-45  1050  0.330
46+     647  0.249

Reincidencia por tramo de antecedentes (train):
                 n   tasa
priors_count             
0             1049  0.205
1-2            971  0.327
3-5            535  0.464
6+             620  0.635

Juvenil (cualquier juv_*_count > 0) (train):
          n   tasa
False  2777  0.335
True    398  0.618

Tipo de cargo (train):
                    n   tasa
c_charge_degree             
F                2038  0.407
M                1137  0.303

Sexo (train):
           n   tasa
sex                
Female   550  0.262
Male    2625  0.393


In [7]:
print("Nº de descripciones distintas en train:", train["c_charge_desc"].nunique())
print("\nTop 40 descripciones por frecuencia (train, sin tasas):")
print(train["c_charge_desc"].value_counts().head(40))

print("\ndays_b_screening_arrest (train):")
sgn = np.sign(train["days_b_screening_arrest"]).map({-1: "negativo", 0: "cero", 1: "positivo"})
print(train.groupby(sgn)["two_year_recid"].agg(n="size", tasa="mean").round(3))

print("\nc_days_from_compas (train):")
print(train["c_days_from_compas"].describe().round(1))
gt0 = train["c_days_from_compas"] > 0
print(train.groupby(gt0)["two_year_recid"].agg(n="size", tasa="mean").round(3))

Nº de descripciones distintas en train: 303

Top 40 descripciones por frecuencia (train, sin tasas):
c_charge_desc
Battery                              485
arrest case no charge                434
Possession of Cocaine                208
Grand Theft in the 3rd Degree        184
Driving While License Revoked        115
Driving Under The Influence           62
Possess Cannabis/20 Grams Or Less     58
Pos Cannabis W/Intent Sel/Del         53
Poss3,4 Methylenedioxymethcath        43
Grand Theft (Motor Vehicle)           39
DUI Property Damage/Injury            39
Felony Battery (Dom Strang)           36
DUI Level 0.15 Or Minor In Veh        36
Burglary Unoccupied Dwelling          36
Susp Drivers Lic 1st Offense          34
Aggravated Assault W/dead Weap        34
Possession of Cannabis                31
Resist/Obstruct W/O Violence          29
Aggrav Battery w/Deadly Weapon        29
Possession Of Alprazolam              28
Burglary Conveyance Unoccup           28
Felony Petit Theft      

In [9]:
def group_charge(desc):
    if pd.isna(desc):
        return "otros"                      # solo 5 nulos en todo el dataset
    d = desc.lower()
    if d == "arrest case no charge":
        return "sin_cargo"
    if any(k in d for k in ["cocaine", "cannabis", "marij", "cathin", "methylenedioxy",
                            "alprazolam", "oxycodone", "hydrocodone", "hydromorph", "heroin",
                            "amphetamine", "narcotic", "controlled sub", "contr subst",
                            "drug", "paraphernalia", "meth"]):
        return "drogas"
    if any(k in d for k in ["theft", "burglary", "larceny", "shoplift", "stolen"]):
        return "robo_hurto"
    if any(k in d for k in ["battery", "assault", "robbery", "murder", "manslaughter",
                            "kidnap", "domestic", "injunc", "dom viol"]):
        return "violencia"
    if any(k in d for k in ["driving", "dui", "licen", "lic ", "susp", "drivers", "operating",
                            "reckless", "leave acc", "leaving acc", "leaving the scene",
                            "lve/scen"]):
        return "conduccion"
    return "otros"

for part in (train, val, test):
    part["charge_group"] = part["c_charge_desc"].map(group_charge)

print("Tamaño de cada grupo (train):")
print(train["charge_group"].value_counts())
print("\nTop 15 descripciones en 'otros' (train, sin tasas):")
print(train.loc[train["charge_group"] == "otros", "c_charge_desc"].value_counts().head(15))

Tamaño de cada grupo (train):
charge_group
violencia     837
drogas        586
otros         454
robo_hurto    446
sin_cargo     434
conduccion    418
Name: count, dtype: int64

Top 15 descripciones en 'otros' (train, sin tasas):
c_charge_desc
Resist/Obstruct W/O Violence        29
Resist Officer w/Violence           22
Crim Use of Personal ID Info        20
Tampering With Physical Evidence    20
Carrying Concealed Firearm          18
Disorderly Conduct                  16
False Imprisonment                  14
Uttering a Forged Instrument        13
Unlaw LicTag/Sticker Attach          9
Criminal Mischief                    9
Neglect Child / No Bodily Harm       9
False Ownership Info/Pawn Item       8
Cruelty Toward Child                 8
Prowling/Loitering                   7
Fleeing Or Attmp Eluding A Leo       7
Name: count, dtype: int64


In [10]:
AGE_BINS, AGE_LABELS = [0, 23, 31, 46, 200], ["<23", "23-30", "31-45", "46+"]
PRIOR_BINS, PRIOR_LABELS = [0, 1, 3, 6, 10_000], ["0", "1-2", "3-5", "6+"]
CHARGE_GROUPS = ["sin_cargo", "drogas", "robo_hurto", "violencia", "conduccion", "otros"]

def dummies(series, categories, reference, prefix):
    """One-hot con categorías FIJAS (no dependen de los datos) y una de referencia omitida."""
    s = series.astype(str)
    return pd.DataFrame({f"{prefix}={c}": (s == c).astype(int)
                         for c in categories if c != reference}, index=series.index)

def make_features(frame):
    age_bin = pd.cut(frame["age"], AGE_BINS, labels=AGE_LABELS, right=False)
    prior_bin = pd.cut(frame["priors_count"], PRIOR_BINS, labels=PRIOR_LABELS, right=False)
    dba = np.sign(frame["days_b_screening_arrest"]).map({-1: "negativo", 0: "cero", 1: "positivo"})

    X = pd.concat([
        (frame["sex"] == "Male").astype(int).rename("is_male"),
        dummies(age_bin, AGE_LABELS, "46+", "age"),
        dummies(prior_bin, PRIOR_LABELS, "0", "priors"),
        (frame[["juv_fel_count", "juv_misd_count", "juv_other_count"]].sum(axis=1) > 0)
            .astype(int).rename("has_juvenile"),
        (frame["c_charge_degree"] == "F").astype(int).rename("is_felony"),
        dummies(frame["charge_group"], CHARGE_GROUPS, "otros", "charge"),
        dummies(dba, ["negativo", "cero", "positivo"], "negativo", "arrest_days"),
        (frame["c_days_from_compas"] > 0).astype(int).rename("days_from_compas_gt0"),
    ], axis=1)
    assert not X.isna().any().any()
    return X

X_train, X_val, X_test = make_features(train), make_features(val), make_features(test)
y_train = train["two_year_recid"].to_numpy()
y_val = val["two_year_recid"].to_numpy()
y_test = test["two_year_recid"].to_numpy()

FEATURE_COLS = list(X_train.columns)
assert list(X_val.columns) == FEATURE_COLS == list(X_test.columns)
assert not any(("race" in c) or ("decile" in c) for c in FEATURE_COLS)   # nunca como feature

print(f"{len(FEATURE_COLS)} features:", FEATURE_COLS)
print("\nTamaños:", X_train.shape, X_val.shape, X_test.shape)

print("\nProporción de 1s por columna:")
print(pd.DataFrame({"train": X_train.mean(), "val": X_val.mean(), "test": X_test.mean()}).round(3))

# Comprobación a posteriori (solo train): tasa de reincidencia por grupo de delito
print("\nReincidencia por grupo de delito (train):")
print(train.groupby("charge_group")["two_year_recid"].agg(n="size", tasa="mean").round(3))

# Guardado de los splits y la lista de features (la app los cargará, no recalcula nada)
SPLITS_DIR = ARTIFACTS_DIR / "splits"
SPLITS_DIR.mkdir(exist_ok=True)
AUDIT_COLS = ["race", "sex", "decile_score", "compas_screening_date"]
for name, part, X, y in [("train", train, X_train, y_train),
                         ("val", val, X_val, y_val),
                         ("test", test, X_test, y_test)]:
    out = X.copy()
    out["y"] = y
    for c in AUDIT_COLS:
        out[c] = part[c].to_numpy()
    out.to_csv(SPLITS_DIR / f"{name}.csv", index=False)
(ARTIFACTS_DIR / "feature_cols.json").write_text(json.dumps(FEATURE_COLS))

17 features: ['is_male', 'age=<23', 'age=23-30', 'age=31-45', 'priors=1-2', 'priors=3-5', 'priors=6+', 'has_juvenile', 'is_felony', 'charge=sin_cargo', 'charge=drogas', 'charge=robo_hurto', 'charge=violencia', 'charge=conduccion', 'arrest_days=cero', 'arrest_days=positivo', 'days_from_compas_gt0']

Tamaños: (3175, 17) (1061, 17) (1061, 17)

Proporción de 1s por columna:
                      train    val   test
is_male               0.827  0.790  0.771
age=<23               0.098  0.127  0.134
age=23-30             0.368  0.327  0.324
age=31-45             0.331  0.333  0.346
priors=1-2            0.306  0.288  0.282
priors=3-5            0.169  0.181  0.153
priors=6+             0.195  0.164  0.151
has_juvenile          0.125  0.122  0.113
is_felony             0.642  0.632  0.590
charge=sin_cargo      0.137  0.119  0.116
charge=drogas         0.185  0.155  0.120
charge=robo_hurto     0.140  0.125  0.159
charge=violencia      0.264  0.310  0.367
charge=conduccion     0.132  0.143  0.1

285

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
import joblib

MODELS_DIR = ARTIFACTS_DIR / "models"
MODELS_DIR.mkdir(exist_ok=True)

def fpr_recall(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    neg, pos = (y_true == 0), (y_true == 1)
    fpr = (y_pred[neg] == 1).mean() if neg.sum() else float("nan")
    rec = (y_pred[pos] == 1).mean() if pos.sum() else float("nan")
    return fpr, rec

def threshold_for_fpr(y_true, scores, limit):
    """Umbral con mayor recall cuya FPR en estos datos es <= limit (positivo si score >= umbral)."""
    fpr, tpr, thr = roc_curve(y_true, scores)
    ok = np.where(fpr <= limit)[0]
    return float(thr[ok[-1]])

# --- Baseline: clase mayoritaria (decisión 5) ---
maj = int(y_train.mean() >= 0.5)
base_pred = np.full(len(y_test), maj)
print("Baseline (clase mayoritaria =", maj, ")")
print("  AUC test:", roc_auc_score(y_test, np.zeros(len(y_test))),
      "| FPR/recall test:", fpr_recall(y_test, base_pred))

# --- Referencia COMPAS: medio/alto = decile_score >= 5 (decisión 4c) ---
COMPAS_CUT = 5
limit_fpr, compas_val_recall = fpr_recall(y_val, (val["decile_score"] >= COMPAS_CUT).astype(int))
compas_test_fpr, compas_test_recall = fpr_recall(y_test, (test["decile_score"] >= COMPAS_CUT).astype(int))
print(f"\nCOMPAS (decile >= {COMPAS_CUT}) en validación: FPR={limit_fpr:.3f} recall={compas_val_recall:.3f}")
print(f"COMPAS en test: FPR={compas_test_fpr:.3f} recall={compas_test_recall:.3f} "
      f"AUC={roc_auc_score(y_test, test['decile_score']):.3f}")

# --- Barrido de C (L2) con AUC en validación (decisión 6) ---
rows = []
for C in [0.01, 0.1, 1, 10]:
    m = LogisticRegression(C=C, max_iter=1000).fit(X_train, y_train)
    rows.append({"C": C, "auc_val": roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])})
sweep = pd.DataFrame(rows)
print("\nBarrido de C:\n", sweep.round(4))

# Regla: la C más pequeña (más regularizada) cuya AUC esté a <= 0.002 de la mejor
best_auc = sweep["auc_val"].max()
best_C = float(sweep.loc[sweep["auc_val"] >= best_auc - 0.002, "C"].min())
print("C elegida:", best_C)

# --- Modelo final sklearn ---
sk = LogisticRegression(C=best_C, max_iter=1000).fit(X_train, y_train)
s_val, s_test = sk.predict_proba(X_val)[:, 1], sk.predict_proba(X_test)[:, 1]

thr = threshold_for_fpr(y_val, s_val, limit_fpr)       # se fija SOLO en validación
for name, y, s in [("val", y_val, s_val), ("test", y_test, s_test)]:
    f, r = fpr_recall(y, (s >= thr).astype(int))
    print(f"sklearn {name}: AUC={roc_auc_score(y, s):.3f}  umbral={thr:.3f}  FPR={f:.3f}  recall={r:.3f}")

print("\nCoeficientes:")
print(pd.Series(sk.coef_[0], index=FEATURE_COLS).sort_values().round(3))
print("intercepto:", sk.intercept_.round(3))

joblib.dump(sk, MODELS_DIR / "sklearn_logreg.joblib")
(ARTIFACTS_DIR / "config.json").write_text(json.dumps(
    {"best_C": best_C, "compas_cut": COMPAS_CUT, "limit_fpr": limit_fpr}));

Baseline (clase mayoritaria = 0 )
  AUC test: 0.5 | FPR/recall test: (np.float64(0.0), np.float64(0.0))

COMPAS (decile >= 5) en validación: FPR=0.265 recall=0.630
COMPAS en test: FPR=0.237 recall=0.530 AUC=0.690

Barrido de C:
        C  auc_val
0   0.01   0.7315
1   0.10   0.7503
2   1.00   0.7507
3  10.00   0.7506
C elegida: 0.1
sklearn val: AUC=0.750  umbral=0.389  FPR=0.265  recall=0.635
sklearn test: AUC=0.729  umbral=0.389  FPR=0.229  recall=0.580

Coeficientes:
charge=sin_cargo       -0.385
charge=conduccion      -0.313
charge=violencia       -0.129
days_from_compas_gt0   -0.045
charge=drogas           0.010
arrest_days=cero        0.038
charge=robo_hurto       0.072
is_felony               0.114
age=31-45               0.159
is_male                 0.302
has_juvenile            0.387
priors=1-2              0.456
age=23-30               0.673
arrest_days=positivo    0.825
priors=3-5              1.025
age=<23                 1.045
priors=6+               1.693
dtype: float64
i

In [12]:
import torch.nn.functional as F

n = len(X_train)
WD = 1.0 / (best_C * n)
LR, EPOCHS = 0.1, 30_000

def to_t(X):
    return torch.tensor(X.to_numpy(dtype="float32"))
Xtr, Xva, Xte = to_t(X_train), to_t(X_val), to_t(X_test)
ytr = torch.tensor(y_train, dtype=torch.float32)

def objective(w, b, X, y):
    """Pérdida media (cross-entropy binaria) + penalización L2 (sin penalizar el intercepto)."""
    z = X @ w + b
    bce = -(y * F.logsigmoid(z) + (1 - y) * F.logsigmoid(-z)).mean()
    return bce + 0.5 * WD * (w ** 2).sum()

w = torch.zeros(Xtr.shape[1], requires_grad=True)
b = torch.zeros(1, requires_grad=True)

history = []
for epoch in range(EPOCHS):
    loss = objective(w, b, Xtr, ytr)
    loss.backward()
    with torch.no_grad():
        w -= LR * w.grad
        b -= LR * b.grad
    w.grad.zero_(); b.grad.zero_()
    if epoch % 2500 == 0 or epoch == EPOCHS - 1:
        history.append((epoch, loss.item()))
print("Pérdida (objetivo) durante el entrenamiento:")
for e, l in history:
    print(f"  época {e:6d}: {l:.6f}")

# Control: el mismo objetivo evaluado en los pesos de sklearn
w_sk = torch.tensor(sk.coef_[0], dtype=torch.float32)
b_sk = torch.tensor(sk.intercept_, dtype=torch.float32)
with torch.no_grad():
    print(f"\nObjetivo manual : {objective(w, b, Xtr, ytr).item():.6f}")
    print(f"Objetivo sklearn: {objective(w_sk, b_sk, Xtr, ytr).item():.6f}")
    p_val = torch.sigmoid(Xva @ w + b).numpy()
    p_test = torch.sigmoid(Xte @ w + b).numpy()

print("\nmax |coef manual - coef sklearn|:", float((w.detach() - w_sk).abs().max()))
print("max |intercepto manual - sklearn|:", float((b.detach() - b_sk).abs().max()))
print("max |prob manual - prob sklearn| (test):", float(np.abs(p_test - s_test).max()))

thr_m = threshold_for_fpr(y_val, p_val, limit_fpr)
for name, y, s in [("val", y_val, p_val), ("test", y_test, p_test)]:
    f, r = fpr_recall(y, (s >= thr_m).astype(int))
    print(f"manual {name}: AUC={roc_auc_score(y, s):.3f}  umbral={thr_m:.3f}  FPR={f:.3f}  recall={r:.3f}")

torch.save({"w": w.detach(), "b": b.detach()}, MODELS_DIR / "manual_torch.pt")

Pérdida (objetivo) durante el entrenamiento:
  época      0: 0.693147
  época   2500: 0.584839
  época   5000: 0.583964
  época   7500: 0.583889
  época  10000: 0.583881
  época  12500: 0.583880
  época  15000: 0.583880
  época  17500: 0.583880
  época  20000: 0.583880
  época  22500: 0.583880
  época  25000: 0.583880
  época  27500: 0.583880
  época  29999: 0.583880

Objetivo manual : 0.583880
Objetivo sklearn: 0.583881

max |coef manual - coef sklearn|: 0.011635780334472656
max |intercepto manual - sklearn|: 0.004125237464904785
max |prob manual - prob sklearn| (test): 0.00360936626532693
manual val: AUC=0.750  umbral=0.389  FPR=0.265  recall=0.635
manual test: AUC=0.729  umbral=0.389  FPR=0.229  recall=0.580


In [13]:
import torch.nn as nn

LR_STD, EPOCHS_STD = 0.1, 5_000

torch.manual_seed(SEED)
model = nn.Linear(Xtr.shape[1], 1)          # w y b, inicializados al azar
loss_fn = nn.BCEWithLogitsLoss()            # sigmoide + cross-entropy, con media
optimizer = torch.optim.Adam(
    [{"params": [model.weight], "weight_decay": WD},    # L2 solo en los pesos
     {"params": [model.bias],   "weight_decay": 0.0}],  # el intercepto no se penaliza
    lr=LR_STD)

history_std = []
for epoch in range(EPOCHS_STD):
    optimizer.zero_grad()
    loss = loss_fn(model(Xtr).squeeze(1), ytr)
    loss.backward()
    optimizer.step()
    if epoch % 500 == 0 or epoch == EPOCHS_STD - 1:
        with torch.no_grad():   # objetivo completo (pérdida + penalización) para comparar
            obj = objective(model.weight.squeeze(0), model.bias, Xtr, ytr).item()
        history_std.append((epoch, obj))

print("Objetivo durante el entrenamiento (Adam):")
for e, o in history_std:
    print(f"  época {e:5d}: {o:.6f}")

with torch.no_grad():
    w_std, b_std = model.weight.squeeze(0), model.bias
    print(f"\nObjetivo estándar: {objective(w_std, b_std, Xtr, ytr).item():.6f}")
    print(f"Objetivo manual  : {objective(w, b, Xtr, ytr).item():.6f}")
    print(f"Objetivo sklearn : {objective(w_sk, b_sk, Xtr, ytr).item():.6f}")
    p_val_std = torch.sigmoid(model(Xva).squeeze(1)).numpy()
    p_test_std = torch.sigmoid(model(Xte).squeeze(1)).numpy()

print("\nmax |coef estándar - coef sklearn|:", float((w_std - w_sk).abs().max()))
print("max |coef estándar - coef manual|:", float((w_std - w.detach()).abs().max()))
print("max |prob estándar - prob sklearn| (test):", float(np.abs(p_test_std - s_test).max()))

thr_s = threshold_for_fpr(y_val, p_val_std, limit_fpr)
for name, y, s in [("val", y_val, p_val_std), ("test", y_test, p_test_std)]:
    f, r = fpr_recall(y, (s >= thr_s).astype(int))
    print(f"estándar {name}: AUC={roc_auc_score(y, s):.3f}  umbral={thr_s:.3f}  FPR={f:.3f}  recall={r:.3f}")

torch.save(model.state_dict(), MODELS_DIR / "standard_torch.pt")

Objetivo durante el entrenamiento (Adam):
  época     0: 0.714078
  época   500: 0.583880
  época  1000: 0.583880
  época  1500: 0.583880
  época  2000: 0.583880
  época  2500: 0.583880
  época  3000: 0.583880
  época  3500: 0.583880
  época  4000: 0.583889
  época  4500: 0.583880
  época  4999: 0.583880

Objetivo estándar: 0.583880
Objetivo manual  : 0.583880
Objetivo sklearn : 0.583881

max |coef estándar - coef sklearn|: 0.011640727519989014
max |coef estándar - coef manual|: 8.764863014221191e-05
max |prob estándar - prob sklearn| (test): 0.003706462231666041
estándar val: AUC=0.750  umbral=0.389  FPR=0.265  recall=0.635
estándar test: AUC=0.729  umbral=0.389  FPR=0.229  recall=0.580


/tmp/ipykernel_3188/1876736513.py:36: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print("\nmax |coef estándar - coef sklearn|:", float((w_std - w_sk).abs().max()))


In [14]:
# ¿Se detuvo sklearn antes de converger? Lo reajustamos con una tolerancia mucho más estricta.
sk_tight = LogisticRegression(C=best_C, max_iter=10_000, tol=1e-12).fit(X_train, y_train)
w_std_np = model.weight.detach().squeeze(0).numpy()
w_man_np = w.detach().numpy()

print("sklearn tol=1e-4 (por defecto) vs tol=1e-12:", np.abs(sk_tight.coef_[0] - sk.coef_[0]).max())
print("sklearn tol=1e-12 vs estándar:", np.abs(sk_tight.coef_[0] - w_std_np).max())
print("sklearn tol=1e-12 vs manual  :", np.abs(sk_tight.coef_[0] - w_man_np).max())

sklearn tol=1e-4 (por defecto) vs tol=1e-12: 0.011583704094737857
sklearn tol=1e-12 vs estándar: 7.75173484970082e-05
sklearn tol=1e-12 vs manual  : 7.07853696617633e-05


In [15]:
models = {"sklearn": (s_val, s_test),
          "manual_torch": (p_val, p_test),
          "standard_torch": (p_val_std, p_test_std)}

rows, thresholds = [], {}
for name, (pv, pt) in models.items():
    t = threshold_for_fpr(y_val, pv, limit_fpr)          # umbral fijado SOLO en val
    thresholds[name] = t
    fv, rv = fpr_recall(y_val, (pv >= t).astype(int))
    ft, rt = fpr_recall(y_test, (pt >= t).astype(int))
    rows.append({"model": name, "threshold": round(t, 3),
                 "AUC_val": roc_auc_score(y_val, pv), "AUC_test": roc_auc_score(y_test, pt),
                 "FPR_val": fv, "recall_val": rv, "FPR_test": ft, "recall_test": rt})

# Referencias: COMPAS (decile >= 5) y baseline de clase mayoritaria
fv, rv = fpr_recall(y_val, (val["decile_score"] >= COMPAS_CUT).astype(int))
ft, rt = fpr_recall(y_test, (test["decile_score"] >= COMPAS_CUT).astype(int))
rows.append({"model": "COMPAS (decile>=5)", "threshold": COMPAS_CUT,
             "AUC_val": roc_auc_score(y_val, val["decile_score"]),
             "AUC_test": roc_auc_score(y_test, test["decile_score"]),
             "FPR_val": fv, "recall_val": rv, "FPR_test": ft, "recall_test": rt})
rows.append({"model": "baseline (clase mayoritaria)", "threshold": np.nan,
             "AUC_val": 0.5, "AUC_test": 0.5,
             "FPR_val": 0.0, "recall_val": 0.0, "FPR_test": 0.0, "recall_test": 0.0})

comparison = pd.DataFrame(rows).set_index("model")
print(comparison.round(3).to_string())

# ¿Predicen lo mismo, persona por persona, en test?
labels = {k: (v[1] >= thresholds[k]).astype(int) for k, v in models.items()}
print("\nAcuerdo de etiquetas en test:")
print("  sklearn vs manual  :", (labels["sklearn"] == labels["manual_torch"]).mean())
print("  sklearn vs estándar:", (labels["sklearn"] == labels["standard_torch"]).mean())
print("  manual  vs estándar:", (labels["manual_torch"] == labels["standard_torch"]).mean())

# Incertidumbre de la ventaja sobre COMPAS en test (bootstrap, solo descriptivo)
rng = np.random.default_rng(SEED)
pred_m = labels["sklearn"]
pred_c = (test["decile_score"].to_numpy() >= COMPAS_CUT).astype(int)
pos, neg = np.where(y_test == 1)[0], np.where(y_test == 0)[0]
d_rec = [pred_m[i].mean() - pred_c[i].mean() for i in (rng.choice(pos, len(pos)) for _ in range(2000))]
d_fpr = [pred_m[i].mean() - pred_c[i].mean() for i in (rng.choice(neg, len(neg)) for _ in range(2000))]
print("\nDiferencia modelo - COMPAS en test, IC95% bootstrap:")
print("  recall:", np.percentile(d_rec, [2.5, 97.5]).round(3))
print("  FPR   :", np.percentile(d_fpr, [2.5, 97.5]).round(3))

# Guardar (la app leerá esto y los modelos; no se reentrena nada)
comparison.to_csv(ARTIFACTS_DIR / "metrics.csv")
cfg = json.loads((ARTIFACTS_DIR / "config.json").read_text())
cfg["thresholds"] = thresholds
(ARTIFACTS_DIR / "config.json").write_text(json.dumps(cfg))
print("\nArchivos en artifacts/:", sorted(str(p.relative_to(ARTIFACTS_DIR)) for p in ARTIFACTS_DIR.rglob("*") if p.is_file()))

                              threshold  AUC_val  AUC_test  FPR_val  recall_val  FPR_test  recall_test
model                                                                                                 
sklearn                           0.389    0.750     0.729    0.265       0.635     0.229         0.58
manual_torch                      0.389    0.750     0.729    0.265       0.635     0.229         0.58
standard_torch                    0.389    0.750     0.729    0.265       0.635     0.229         0.58
COMPAS (decile>=5)                5.000    0.737     0.690    0.265       0.630     0.237         0.53
baseline (clase mayoritaria)        NaN    0.500     0.500    0.000       0.000     0.000         0.00

Acuerdo de etiquetas en test:
  sklearn vs manual  : 1.0
  sklearn vs estándar: 1.0
  manual  vs estándar: 1.0

Diferencia modelo - COMPAS en test, IC95% bootstrap:
  recall: [-0.003  0.105]
  FPR   : [-0.047  0.029]

Archivos en artifacts/: ['config.json', 'feature_cols.json',